# Feature Extraction with DINO

## Purpose

Try using DINO's feature space to perform segmentation

## Process

1. Load the raw TIF file
2. Pick one of the layers
3. Preprocess the image
    - Non-negative values
    - Denoise
4. Run through DINOv2
5. Analyze feature space

In [1]:
from skimage.io import imread
import stackview
import numpy as np

: 

In [ ]:
image_path = "D:/JHU/BDD/Spine Detect/Data/Gaby/Example/F13_2_20250323_roi1_Red.tif"
image = imread(image_path, plugin='tifffile')

stackview.slice(image)
stackview.picker(image)
# stackview.display_range(image)
stackview.histogram(image)

In [ ]:
from skimage.exposure import equalize_adapthist
image = np.array([equalize_adapthist(image[i,:,:]) for i in range(image.shape[0])])
stackview.histogram(image)

In [ ]:
# Calculate the min and max values that we want to use
# Analyze the distribution of pixel intensities, and choose a range that contains 95% of the pixel values
vmin = np.percentile(image, 2.5)
vmax = np.percentile(image, 99)

vmin=0

# Resample the image so that pixels are in [0, 1] range
image_rescaled = (image - vmin) / (vmax - vmin)
image_rescaled = np.clip(image_rescaled, 0, 1)
image_rescaled = (image_rescaled * 1024).astype(np.uint16)
# stackview.display_range(image_rescaled)
stackview.histogram(image_rescaled)

In [ ]:
# from skimage.filters import gaussian

# anisotropy = 0.096
# sigma = 2 * np.array([1, 1, anisotropy])
# image_denoised = gaussian(image_rescaled, sigma=sigma, preserve_range=True)
# stackview.histogram(image_denoised)

In [ ]:
# from skimage.filters import median

# # anisotropy = 0.096
# # sigma = 2 * np.array([1, 1, anisotropy])
# image_denoised = median(image_rescaled[45, :, :])
# stackview.histogram(image_denoised)

In [ ]:
# from skimage.restoration import denoise_wavelet

# # anisotropy = 0.096
# # sigma = 2 * np.array([1, 1, anisotropy])
# image_denoised = denoise_wavelet(image_rescaled, channel_axis=0, wavelet='sym9')
# stackview.histogram(image_denoised)

In [ ]:
# import pywt

# # families = [
# #     'haar',
# #     'db',
# #     'sym',
# #     'coif',
# #     'bior',
# #     'rbio',
# #     'dmey',
# # ]

# filtered = [
#     denoise_wavelet(image_rescaled[45,:,:], wavelet=wavelet_name)
#     for wavelet_name in pywt.wavelist(kind='discrete')
# ]

# # pywt.families()

In [ ]:
# stackview.slice(np.array(filtered))

In [ ]:
from skimage.filters import median
from skimage.restoration import denoise_wavelet
import pywt

print(pywt.wavelist(kind='discrete')[87])
denoised_image = denoise_wavelet(image_rescaled, channel_axis=0, wavelet=pywt.wavelist(kind='discrete')[87])
filtered_image = np.array([
    median(denoised_image[i,:,:])
    for i in range(denoised_image.shape[0])
])
# rescale the filtered image to [0, 1024]
filtered_image = (filtered_image - filtered_image.min()) / (filtered_image.max() - filtered_image.min())
filtered_image = (filtered_image * 1024).astype(np.uint16)
stackview.histogram(filtered_image)

In [ ]:
from skimage.restoration import estimate_sigma

sigma_est = np.mean(estimate_sigma(filtered_image, channel_axis=0))

min_val = np.mean(filtered_image) + sigma_est
print(f"Estimated noise standard deviation = {sigma_est}, min_val = {min_val}")

min_val = np.percentile(filtered_image, 50)
print(f"Using min_val = {min_val}")

threshold_image = np.clip(filtered_image, min_val, None)
threshold_image = threshold_image - min_val
stackview.histogram(threshold_image)

In [ ]:
# from skimage.exposure import equalize_adapthist

# stackview.histogram(equalize_adapthist(threshold_image[47,:,:]))

In [ ]:
# save the image
from skimage.io import imsave
# rescale to [0, 1024]
threshold_image = (threshold_image - threshold_image.min()) / (threshold_image.max() - threshold_image.min())
threshold_image = (threshold_image * 1024).astype(np.uint16)
imsave("D:/JHU/BDD/Spine Detect/Data/Gaby/Example/denoised.tif", threshold_image)

In [ ]:
# save just one slice as a png
stackview.imshow(threshold_image[45,:,:])

In [ ]:
# # Start by denoising the image
# from skimage.restoration import denoise_nl_means, estimate_sigma
# sigma_est = np.mean(estimate_sigma(image_rescaled))
# print(f"Estimated noise standard deviation = {sigma_est}")

In [ ]:
import torch

# DINOv2
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
model.eval().to('cuda')

In [ ]:
_PATCH = 14
_IMAGENET_MEAN = (0.485, 0.456, 0.406)
_IMAGENET_STD  = (0.229, 0.224, 0.225)

In [ ]:
import torch.nn.functional as F

def _to_chw_normalized(img: np.ndarray) -> torch.Tensor:
    """
    img: np.ndarray, shape (H, W) or (H, W, 3), dtype uint8/float32 in [0,255] or [0,1].
    Returns torch.Tensor (C, H, W), float32, ImageNet normalized.
    """
    if img.ndim == 2:  # grayscale
        img = np.stack([img, img, img], axis=-1)
    assert img.ndim == 3 and img.shape[2] == 3, "Expected HxW or HxWx3 array."

    arr = img.astype(np.float32)
    if arr.max() > 1.0:
        arr = arr / 255.0
    mean = np.array(_IMAGENET_MEAN, dtype=np.float32)
    std  = np.array(_IMAGENET_STD, dtype=np.float32)
    arr = (arr - mean) / std                       # H W C
    arr = np.transpose(arr, (2, 0, 1))             # C H W
    return torch.from_numpy(arr)                   # C H W

def _pad_to_multiple(x_chw: torch.Tensor, m: int = _PATCH):
    """
    Pads CHW so H,W are multiples of m using 'replicate' border.
    Returns (x_padded, (pad_h, pad_w)).
    """
    _, H, W = x_chw.shape
    pad_h = (m - (H % m)) % m
    pad_w = (m - (W % m)) % m
    if pad_h == 0 and pad_w == 0:
        return x_chw, (0, 0)
    x = F.pad(x_chw.unsqueeze(0), (0, pad_w, 0, pad_h), mode='replicate').squeeze(0)
    return x, (pad_h, pad_w)

@torch.no_grad()
def _extract_patch_tokens(model, x_bchw: torch.Tensor) -> torch.Tensor:
    """
    Returns patch tokens [B, N, D] (CLS removed).
    Uses torch.hub DINOv2 API: forward_features dict or get_intermediate_layers fallback.
    """
    # Try forward_features first (covers most versions)
    try:
        out = model.forward_features(x_bchw)
        if isinstance(out, dict):
            # Common keys vary between releases
            for k in ('x_norm_patchtokens', 'x_patch', 'x_tokens', 'x'):
                if k in out:
                    t = out[k]  # [B, N, D] or [B, N+1, D]
                    if t.dim() != 3:
                        raise RuntimeError("Unexpected token tensor rank.")
                    n_grid = (x_bchw.shape[-2] // _PATCH) * (x_bchw.shape[-1] // _PATCH)
                    if t.size(1) == n_grid + 1:  # includes CLS
                        t = t[:, 1:, :]
                    return t
        elif isinstance(out, torch.Tensor):  # usually [B, N+1, D]
            return out[:, 1:, :]
    except Exception:
        pass

    # Fallback: last layer tokens
    layers = model.get_intermediate_layers(x_bchw, n=1, return_class_token=True)
    tokens = layers[-1][0]  # [B, N+1, D], CLS first
    return tokens[:, 1:, :]

In [ ]:
x_chw = _to_chw_normalized(threshold_image[45, :, :])           # C,H,W
x_chw, pad_hw = _pad_to_multiple(x_chw, _PATCH)
_, H, W = x_chw.shape
H_grid, W_grid = H // _PATCH, W // _PATCH

x_bchw = x_chw.unsqueeze(0).cuda()            # 1,C,H,W
tokens = _extract_patch_tokens(model, x_bchw) # 1,N,D
tokens = tokens.squeeze(0).cpu().numpy()      # N,D
grid = tokens.reshape(H_grid, W_grid, -1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.cluster import MiniBatchKMeans

def _l2_normalize(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """
    Row-wise L2 normalization -> approximates cosine similarity usage.
    X: (N, D)
    """
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / (n + eps)

def kmeans_cluster_embeddings(
    embeddings: np.ndarray,
    grid_hw: tuple[int, int],
    k: int = 3,
    l2_normalize: bool = True,
    batch_size: int = 4096,
    max_iter: int = 100,
    random_state: int = 0,
) -> np.ndarray:
    """
    embeddings: (N, D) in row-major patch order
    grid_hw: (H_grid, W_grid)
    Returns labels_grid: (H_grid, W_grid) with integer labels [0..k-1]
    """
    Hg, Wg = grid_hw
    assert embeddings.shape[0] == Hg * Wg, "Mismatch between N and grid size."
    X = _l2_normalize(embeddings) if l2_normalize else embeddings

    km = MiniBatchKMeans(
        n_clusters=k,
        batch_size=min(batch_size, X.shape[0]),
        max_iter=max_iter,
        n_init="auto",
        random_state=random_state,
        verbose=False,
    )
    labels = km.fit_predict(X)  # (N,)
    return labels.reshape(Hg, Wg)

def show_clusters_imshow(
    labels_grid: np.ndarray,
    title: str | None = "K-means clusters",
    ax: plt.Axes | None = None,
    cmap_name: str = "tab20",
    vmin: float | None = None,
    vmax: float | None = None,
    show_colorbar: bool = False,
):
    """
    Display integer cluster labels with a discrete colormap.
    """
    k = int(labels_grid.max()) + 1
    # Build a discrete colormap with at least k colors.
    base_cmap = plt.get_cmap(cmap_name)
    # For tab10/tab20, we can sample directly; otherwise, generate from base.
    colors = [base_cmap(i % base_cmap.N) for i in range(k)]
    cmap = ListedColormap(colors, name=f"{cmap_name}_{k}")

    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(labels_grid, interpolation="nearest", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title)
    if show_colorbar:
        # Draw a colorbar with discrete ticks at integer labels.
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_ticks(np.arange(k) + 0.5)  # center ticks
        cbar.set_ticklabels([str(i) for i in range(k)])
    plt.tight_layout()
    return ax

In [ ]:
import imageio.v3 as iio
import matplotlib.pyplot as plt

labels_grid = kmeans_cluster_embeddings(
    embeddings=tokens,
    grid_hw=(H_grid, W_grid),
    k=12,                    # start with 3: {spines, dendrites, background}
    l2_normalize=True,
    batch_size=4096*2,
    max_iter=1000,
    random_state=0,
)

fig, axs = plt.subplots(1, 2, figsize=(12, 6))
axs[0].imshow(threshold_image[45, :, :], cmap='gray')
show_clusters_imshow(labels_grid, title="DINOv2 patch clusters (K=3)", show_colorbar=True, ax=axs[1])
plt.show()